# 06. 과속방지턱 전처리

전국 과속방지턱 원본을 229개 행정구역 기준으로 정리하고,
관리번호와 좌표가 동일한 중복 설치지점을 제거해 지역별 개수를 집계합니다.

**출력**
- `../data_processed/speed_bump_count_final.csv`

In [1]:
import pandas as pd

speed_bump = pd.read_csv(
    "../data_raw/과속방지턱정보.csv",
    encoding="cp949",
    low_memory=False
)

print("원본:", speed_bump.shape)
display(speed_bump.head())

원본: (141140, 27)


,개방자치단체코드,관리번호,과속방지턱관리번호,도로명,시도명,시군구명,소재지도로명주소,소재지지번주소,설치장소,과속방지턱재료,...,WGS84경도,보차분리여부,연속형여부,과속방지턱설치연도,관리기관명,관리기관전화번호,데이터기준일자,데이터갱신구분,데이터갱신시점,최종수정시점
0,3010000,202130100000601790,NaN,신당동,서울특별시,중구,서울특별시 중구 동호로18길 20,서울특별시 중구 신당동 415-12,도로,원호형,...,127.009382,N,N,NaN,서울특별시 중구청,02-3396-6112,2021-08-11,I,2026-05-15 18:45:22,2021-08-11 13:09:58
1,3010000,202130100000601563,NaN,황학동,서울특별시,중구,서울특별시 중구 서애로1길 47,서울특별시 중구 묵정동 29-31,도로,원호형,...,127.0006646,N,N,NaN,서울특별시 중구청,02-3396-6112,2021-08-11,I,2026-05-15 18:45:22,2021-08-11 13:09:56
2,3010000,202130100000601780,NaN,신당동,서울특별시,중구,서울특별시 중구 동호로20길 76,서울특별시 중구 신당동 398-9,도로,원호형,...,127.009888,N,N,NaN,서울특별시 중구청,02-3396-6112,2021-08-11,I,2026-05-15 18:45:22,2021-08-11 13:09:57
3,3010000,202130100000602142,NaN,태평로1가,서울특별시,중구,서울특별시 중구 다산로36길 79,서울특별시 중구 신당동 44-27,도로,원호형,...,127.0188884,N,N,NaN,서울특별시 중구청,02-3396-6112,2021-08-11,I,2026-05-15 18:45:22,2021-08-11 13:10:01
4,3010000,202330100000600004,NaN,서울특별시 중구 청구로4길,서울특별시,중구,서울특별시 중구 청구로4길 30,서울특별시 중구 신당동 290-90,도로,원호형,...,127.015606,N,N,2021.0,서울특별시 중구,02-3396-6112,2023-03-30,I,2026-05-15 18:45:22,2023-03-17 14:20:12


## 1) 지역명 정규화

In [2]:
sb = speed_bump.copy()

sb["SIDO_NM"] = sb["시도명"].astype(str).str.strip()
sb["SIGUNGU_NM"] = sb["시군구명"].astype(str).str.strip()

sb["SIDO_NM"] = sb["SIDO_NM"].replace({
    "강원도": "강원특별자치도",
    "전라북도": "전북특별자치도"
})

parent_cities = [
    "수원시", "성남시", "안양시", "부천시",
    "안산시", "고양시", "용인시",
    "천안시", "청주시", "전주시",
    "포항시", "창원시"
]

for city in parent_cities:
    mask = sb["SIGUNGU_NM"].str.replace(
        " ", "", regex=False
    ).str.startswith(city, na=False)

    sb.loc[mask, "SIGUNGU_NM"] = city

print("지역 조합:", sb[["SIDO_NM", "SIGUNGU_NM"]].drop_duplicates().shape[0])

지역 조합: 171


## 2) 동일 설치지점 중복 제거 및 집계

In [3]:
sb_unique = sb.drop_duplicates(
    subset=[
        "SIDO_NM",
        "SIGUNGU_NM",
        "과속방지턱관리번호",
        "WGS84위도",
        "WGS84경도"
    ]
).copy()

speed_bump_count_final = (
    sb_unique
    .groupby(["SIDO_NM", "SIGUNGU_NM"])
    .size()
    .reset_index(name="speed_bump_count")
)

print("원본 행:", len(sb))
print("중복 정리 후 설치지점:", len(sb_unique))
print("데이터 제공 지역:", len(speed_bump_count_final))
display(speed_bump_count_final.head())

원본 행: 141140
중복 정리 후 설치지점: 103745
데이터 제공 지역: 171


,SIDO_NM,SIGUNGU_NM,speed_bump_count
0,강원특별자치도,강릉시,500
1,강원특별자치도,동해시,1040
2,강원특별자치도,삼척시,361
3,강원특별자치도,양양군,222
4,강원특별자치도,영월군,610


In [4]:
speed_bump_count_final.to_csv(
    "../data_processed/speed_bump_count_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료
